In [ ]:
# Copy-paste this single cell into your notebook (Jupyter / VS Code / Voila)
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------------
# Helpers
# -------------------------------
def get_uploaded_file_content(upload_widget):
    """Safely extract bytes from FileUpload widget (dict or tuple formats)."""
    if not upload_widget.value:
        return None
    val = upload_widget.value
    if isinstance(val, dict):
        return list(val.values())[0]["content"]
    if isinstance(val, tuple):
        return val[0]["content"]
    return None

def normalize_image(img):
    """Normalize any array to uint8 (0–255) for display."""
    img = img - np.min(img)
    if np.max(img) > 0:
        img = img / np.max(img)
    return (img * 255).astype(np.uint8)

def resize_if_needed(pil_img, max_size=512):
    """Resize using thumbnail() while keeping aspect-ratio if needed."""
    if pil_img.width > max_size or pil_img.height > max_size:
        pil_img = pil_img.copy()
        pil_img.thumbnail((max_size, max_size))
    return pil_img

def to_grayscale_array(pil_img):
    """Return a float32 grayscale numpy array for FFT processing."""
    if pil_img.mode != "L":
        pil_img = pil_img.convert("L")
    return np.array(pil_img, dtype=np.float32)

def circular_mask(shape, radius, filter_type):
    """Return mask for None/Low-pass/High-pass. shape=(rows,cols)."""
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    Y, X = np.ogrid[:rows, :cols]
    dist = np.sqrt((X - ccol)**2 + (Y - crow)**2)
    if filter_type == 'Low-pass':
        mask = dist <= radius
    elif filter_type == 'High-pass':
        mask = dist >= radius
    else:  # 'None'
        mask = np.ones_like(dist, dtype=bool)
    return mask.astype(float)

# -------------------------------
# Widgets
# -------------------------------
upload = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload Image",
    style={'description_width': 'initial'}
)

filter_type = widgets.Dropdown(
    options=['None', 'Low-pass', 'High-pass'],
    value='Low-pass',
    description='Filter:'
)

radius_slider = widgets.IntSlider(
    value=40, min=1, max=400, step=1,
    description="Radius (px):", continuous_update=False
)

fft_scale = widgets.FloatSlider(
    value=1.0, min=0.05, max=3.0, step=0.05,
    description="FFT display scale:", continuous_update=False
)

cmap_selector = widgets.Dropdown(
    options=['gray','magma','inferno','viridis'],
    value='magma',
    description='Colormap:'
)

out_status = widgets.Output()
out_original = widgets.Output()
out_fft = widgets.Output()
out_filtered = widgets.Output()

# -------------------------------
# Processing & Display
# -------------------------------
def process_and_display(change=None):
    # Clear outputs
    out_status.clear_output(wait=True)
    out_original.clear_output(wait=True)
    out_fft.clear_output(wait=True)
    out_filtered.clear_output(wait=True)

    content = get_uploaded_file_content(upload)
    if content is None:
        with out_status:
            print("Waiting for image upload...")
        return

    try:
        with out_status:
            print("Loading image...")

        pil_img = Image.open(io.BytesIO(content))
        pil_img = resize_if_needed(pil_img, max_size=512)
        gray_arr = to_grayscale_array(pil_img)

        with out_status:
            print(f"Image size (w×h): {pil_img.size}. Computing FFT...")

        # 2D FFT
        F = np.fft.fft2(gray_arr)
        Fshift = np.fft.fftshift(F)
        magnitude = np.abs(Fshift)
        # display-magnitude scaled (user scale applied)
        display_magnitude = np.log1p(magnitude) * fft_scale.value

        # Create mask according to selected filter type and radius
        rad = radius_slider.value
        mask = circular_mask(gray_arr.shape, rad, filter_type.value)
        Fshift_filtered = Fshift * mask

        # Inverse FFT back to image
        F_ishift = np.fft.ifftshift(Fshift_filtered)
        img_filtered = np.abs(np.fft.ifft2(F_ishift))
        img_filtered_norm = normalize_image(img_filtered)

        # ---------------- Display original ----------------
        with out_original:
            plt.figure(figsize=(3.8,3.8))
            # If input was color, show the color image; else show grayscale
            try:
                # attempt to display original color if present
                orig_img_display = pil_img if pil_img.mode in ("RGB","RGBA") else Image.fromarray(gray_arr.astype(np.uint8))
            except Exception:
                orig_img_display = Image.fromarray(gray_arr.astype(np.uint8))
            plt.imshow(orig_img_display, cmap=None if pil_img.mode in ("RGB","RGBA") else cmap_selector.value)
            plt.title("Original Image")
            plt.axis('off')
            plt.show()

        # ---------------- Display FFT magnitude ----------------
        with out_fft:
            plt.figure(figsize=(3.8,3.8))
            plt.imshow(display_magnitude, cmap=cmap_selector.value)
            plt.title(f"FFT Magnitude (scale ×{fft_scale.value:.2f})")
            plt.axis('off')
            plt.show()

        # ---------------- Display filtered image ----------------
        with out_filtered:
            plt.figure(figsize=(3.8,3.8))
            plt.imshow(img_filtered_norm, cmap='gray')
            plt.title(f"Filtered Image ({filter_type.value}, r={rad}px)")
            plt.axis('off')
            plt.show()

    except Exception as e:
        with out_status:
            print("Error processing image:")
            print(e)

# -------------------------------
# Observers / Callbacks
# -------------------------------
upload.observe(process_and_display, names='value')
filter_type.observe(process_and_display, names='value')
radius_slider.observe(process_and_display, names='value')
fft_scale.observe(process_and_display, names='value')
cmap_selector.observe(process_and_display, names='value')

# -------------------------------
# Layout
# -------------------------------
controls = widgets.VBox([
    widgets.HBox([upload, filter_type, radius_slider]),
    widgets.HBox([fft_scale, cmap_selector]),
    out_status
])

display(widgets.VBox([
    controls,
    widgets.HBox([out_original, out_fft, out_filtered])
]))
